In [8]:
import pandas as pd
import numpy as np
import time

# ==============================================================================
# 🕵️ EXPERIMENT: UNMASKING THE GROUPBY OBJECT & THE 'SORT' PENALTY
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. THE HASH MAP ILLUSION (No data is actually split)
# ------------------------------------------------------------------------------
# Let's create a tiny groupby object to see its internal C-level structure
small_df = pd.DataFrame({'Cat': ['X', 'Y', 'X', 'Y', 'Z'], 'Val': [1, 2, 3, 4, 5]})
grouped = small_df.groupby('Cat')

print("--- 1. The Hidden Grouping Dictionary ---")
# The .groups attribute reveals the internal Hash Map (Pointer Dictionary)
print(grouped.groups)
print("Notice: It just stores the 'Row Indices' for each group, NOT the actual data!\n")

# ------------------------------------------------------------------------------
# 2. THE SORTING PENALTY (CPU Time Wasted)
# ------------------------------------------------------------------------------
print("--- 2. Benchmarking The Sort Penalty ---")
# Generating a massive 10 Million rows dataset for benchmarking
N = 10_000_000
df = pd.DataFrame({
    'Category': np.random.choice(['A', 'B', 'C', 'D', 'E', 'F'], size=N),
    'Value': np.random.rand(N)
})

# Default GroupBy (sort=True by default)
start_time = time.time()
result_default = df.groupby('Category').sum()
time_default = time.time() - start_time
print(f"Time taken (Default, sort=True): {time_default:.4f} seconds")

# Optimized GroupBy (sort=False)
start_time = time.time()
result_fast = df.groupby('Category', sort=False).sum()
time_fast = time.time() - start_time
print(f"Time taken (Optimized, sort=False): {time_fast:.4f} seconds")

# ==============================================================================
# CONCLUSION
# ==============================================================================
print("-" * 50)
speedup_percentage = ((time_default - time_fast) / time_default) * 100
print(f"🚀 Speedup: Bypassing the sorting phase saved ~{speedup_percentage:.2f}% of CPU execution time!")

--- 1. The Hidden Grouping Dictionary ---
{'X': [0, 2], 'Y': [1, 3], 'Z': [4]}
Notice: It just stores the 'Row Indices' for each group, NOT the actual data!

--- 2. Benchmarking The Sort Penalty ---
Time taken (Default, sort=True): 0.3185 seconds
Time taken (Optimized, sort=False): 0.3983 seconds
--------------------------------------------------
🚀 Speedup: Bypassing the sorting phase saved ~-25.04% of CPU execution time!
